# 📺 YouTube News Bot — Google Colab Runner

Run your YouTube News Bot from Google Colab. Files are stored in Google Drive so nothing is lost between sessions.

**Steps:**
1. Mount Google Drive
2. Clone / update your repo
3. Install dependencies
4. Set your secrets
5. Run the pipeline or dashboard

## ① Mount Google Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BOT_DIR = '/content/drive/MyDrive/youtube-news-bot'
os.makedirs(BOT_DIR, exist_ok=True)
print(f'✅ Google Drive mounted. Bot folder: {BOT_DIR}')

## ② Clone or Update the Repo

In [ ]:
import os

REPO_URL = 'https://github.com/harijothivenkatraman/newsyt.git'
REPO_DIR = '/content/newsyt'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

## ③ Install Dependencies

In [ ]:
# Install CPU-only PyTorch first (smaller, faster)
!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install -r requirements.txt -q
!pip install pyngrok -q
print('✅ Dependencies installed!')

## ④ Download AI Models

Models are saved to Google Drive so they only download **once** across sessions.

In [ ]:
import os, shutil
from pathlib import Path

DRIVE_MODELS = '/content/drive/MyDrive/youtube-news-bot/models'
LOCAL_MODELS = '/content/newsyt/models'

os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

# Symlink Drive models folder into the repo so the bot finds them
if not os.path.islink(LOCAL_MODELS):
    shutil.rmtree(LOCAL_MODELS, ignore_errors=True)
    os.symlink(DRIVE_MODELS, LOCAL_MODELS)
    print(f'✅ models/ -> Google Drive ({DRIVE_MODELS})')
else:
    print(f'✅ models/ already linked to Drive')

# Download piper binary (for English TTS)
import urllib.request, zipfile
piper_exe = '/content/newsyt/bin/piper/piper'
if not os.path.exists(piper_exe):
    print('Downloading Piper TTS binary...')
    urllib.request.urlretrieve(
        'https://github.com/rhasspy/piper/releases/download/2023.11.14-2/piper_linux_x86_64.tar.gz',
        '/tmp/piper.tar.gz'
    )
    os.makedirs('/content/newsyt/bin', exist_ok=True)
    !tar -xzf /tmp/piper.tar.gz -C /content/newsyt/bin/
    os.chmod(piper_exe, 0o755)
    print('✅ Piper binary ready')
else:
    print('✅ Piper binary already exists')

# Download Piper voice model (cached to Drive)
model_onnx = f'{DRIVE_MODELS}/en_US-lessac-medium.onnx'
model_json = f'{DRIVE_MODELS}/en_US-lessac-medium.onnx.json'
base_url = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium'

if not os.path.exists(model_onnx):
    print('Downloading Piper voice model (~63 MB)...')
    urllib.request.urlretrieve(f'{base_url}/en_US-lessac-medium.onnx', model_onnx)
    urllib.request.urlretrieve(f'{base_url}/en_US-lessac-medium.onnx.json', model_json)
    print('✅ Voice model downloaded to Drive')
else:
    print('✅ Voice model already on Drive')

print('\n🎉 All models ready!')

## ⑤ Set Secrets

Upload your `client_secrets.json` and `youtube_token.pickle` from Google Drive.

In [ ]:
import os, shutil

# ─── EDIT THESE ────────────────────────────────────────────────
CHANNEL_NAME   = 'My News Channel'     # Your YouTube channel name
GEMINI_API_KEY = ''                     # Optional: only if USE_LOCAL_ML=false
USE_LOCAL_ML   = 'true'                # 'true' = Flan-T5 offline | 'false' = Gemini API
TTS_ENGINE     = 'piper'               # 'piper' (best) or 'gtts'
# ───────────────────────────────────────────────────────────────

os.environ['CHANNEL_NAME']   = CHANNEL_NAME
os.environ['USE_LOCAL_ML']   = USE_LOCAL_ML
os.environ['TTS_ENGINE']     = TTS_ENGINE
os.environ['VOICE_LANGUAGE'] = 'en'
os.environ['OUTPUT_DIR']     = '/content/newsyt/output'
os.environ['LOG_DIR']        = '/content/drive/MyDrive/youtube-news-bot/logs'
os.environ['VIDEO_RESOLUTION'] = '1280x720'
os.environ['MAX_ARTICLES_PER_RUN'] = '3'

if GEMINI_API_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

os.makedirs(os.environ['OUTPUT_DIR'], exist_ok=True)
os.makedirs(os.environ['LOG_DIR'], exist_ok=True)

# Copy credentials from Drive (if they exist there)
DRIVE_BOT = '/content/drive/MyDrive/youtube-news-bot'
for fname in ['client_secrets.json', 'youtube_token.pickle']:
    src = f'{DRIVE_BOT}/{fname}'
    dst = f'/content/newsyt/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'✅ Copied {fname} from Drive')
    elif os.path.exists(dst):
        print(f'✅ {fname} already in place')
    else:
        print(f'⚠️  {fname} not found in Drive — upload it to {DRIVE_BOT}/')

print('\n✅ Environment configured!')

## ⑥ Run Pipeline Once (manual trigger)

In [ ]:
import os
os.chdir('/content/newsyt')

# dry_run=True → generates content but doesn't upload to YouTube
# dry_run=False → actually uploads
DRY_RUN = True

!python pipeline.py --once {'--dry-run' if DRY_RUN else ''}

## ⑦ Run Dashboard with Public URL (via ngrok)

This starts the full dashboard with a public URL you can open in your browser.

In [ ]:
# Get a free ngrok auth token from https://dashboard.ngrok.com/signup
NGROK_TOKEN = ''  # ← Paste your ngrok token here

import os, threading
os.chdir('/content/newsyt')

if not NGROK_TOKEN:
    print('⚠️  Add your NGROK_TOKEN above to get a public URL')
    print('   Get one free at: https://dashboard.ngrok.com/signup')
else:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    # Start the dashboard in background
    def run_dashboard():
        os.system('python dashboard/app.py')

    t = threading.Thread(target=run_dashboard, daemon=True)
    t.start()

    import time
    time.sleep(3)  # Wait for Flask to start

    port = int(os.getenv('DASHBOARD_PORT', 5050))
    tunnel = ngrok.connect(port)
    print(f'\n🌐 Dashboard URL: {tunnel.public_url}')
    print('   Open this link in your browser!')

## ⑧ Save YouTube Token back to Drive

Run this after logging in via the dashboard to save your token to Drive for next time.

In [ ]:
import shutil, os

DRIVE_BOT = '/content/drive/MyDrive/youtube-news-bot'
for fname in ['youtube_token.pickle', 'client_secrets.json']:
    src = f'/content/newsyt/{fname}'
    dst = f'{DRIVE_BOT}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Saved {fname} to Drive')
    else:
        print(f'⚠️  {fname} not found locally')